# Klasifikasi Fakta/Hoaks berbasis Kalimat & Ekstraksi Entitas (NER)

Arsitektur ini berfokus pada efisiensi tinggi menggunakan Google Colab GPU T4.
Terdiri dari dua arsitektur paralel:
1.  **IndoBERT Fine-Tuned (Trainer)**: Mendeteksi teks pada level *Sentence Classification*.
2.  **NER Pipeline Eksternal**: Menangkap entitas nama/lokasi pada kalimat untuk melengkapi konteks deteksi hoaks.
*Catatan: Augmentasi minoritas hanya diterapkan pada set latih (train-only), tidak pada validasi/uji.*


In [1]:
# 1. Instalasi Dependensi Inti
!pip install -q transformers datasets accelerate sentencepiece scikit-learn nltk

In [2]:
# 2. Pengaitan (Mounting) Google Drive
# Wajib dijalankan untuk mendapatkan akses ke folder scrape_output
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 3. Impor Library dan Persiapan Lingkungan GPU
import os
import gc
import shutil
import warnings
import re
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    set_seed,
    pipeline,
)

import nltk
from nltk.tokenize import sent_tokenize
from google.colab import files

warnings.filterwarnings("ignore")
nltk.download("punkt")
nltk.download("punkt_tab")

# Mematikan thread konversi safetensors otomatis di latar belakang
os.environ["HF_HUB_DISABLE_IMPLICIT_SAFETENSORS"] = "1"

print("Library dan Environment berhasil diinisialisasi.")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Library dan Environment berhasil diinisialisasi.


In [4]:
# 4. Konfigurasi Arsitektur (Teroptimasi GPU T4 - 15GB VRAM)
@dataclass
class Config:
    # Path absolut menuju Google Drive Anda
    data_dir: str = "/content/drive/MyDrive/scrape_output/"
    file_hoaks: str = "data_hoaks_turnbackhoaks.csv"
    file_nonhoaks_cnn: str = "data_nonhoaks_cnn.csv"
    file_nonhoaks_detik: str = "data_nonhoaks_detik.csv"
    file_nonhoaks_kompas: str = "data_nonhoaks_kompas.csv"

    model_name: str = "indolem/indobert-base-uncased"
    ner_model_name: str = "cahya/bert-base-indonesian-NER"

    # Strategi representasi teks
    text_strategy: str = "judul_summary"
    summary_min_words: int = 8
    summary_max_words: int = 96
    fallback_num_sentences: int = 3
    fallback_max_words: int = 96
    max_length: int = 192
    dedup_strategy: str = "url_then_judul_tanggal_then_text_label"

    # Konfigurasi imbalance
    imbalance_method: str = "class_weight"  # opsi: class_weight | focal
    focal_gamma: float = 2.0

    # Konfigurasi augmentasi minoritas
    augment_enable: bool = True
    augment_target_ratio: float = 1.0
    augment_entity_types: Tuple[str, ...] = ("PER", "ORG", "LOC")
    augment_max_per_sample: int = 1
    augment_max_rounds: int = 6
    augment_fallback_eda_prob: float = 0.05
    augment_seed: int = 42

    # Optimasi VRAM T4
    train_batch_size: int = 16
    eval_batch_size: int = 32
    grad_accumulation: int = 2
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    num_epochs: int = 3
    auto_find_batch_size: bool = True
    early_stopping_patience: int = 2
    early_stopping_threshold: float = 0.0

    # Konfigurasi NER inferensi
    ner_infer_device_default: int = -1
    ner_infer_use_gpu: bool = False
    ner_safe_offload_classifier: bool = True

    seed: int = 42
    output_dir: str = "indobert_hoax_ner_model"

cfg = Config()
set_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    torch.cuda.empty_cache()
    gc.collect()

print(f"Target komputasi: {device}")


Target komputasi: cuda


In [5]:
# 5. Data Loading, Prapemrosesan, dan Augmentasi Minoritas (Hanya Set Latih)
BASE_COLS = ["url", "judul", "tanggal", "isi_berita", "Narasi", "Clean Narasi", "hoax", "summary"]
TEXT_LEN_CANDIDATES = ["Clean Narasi", "Narasi", "isi_berita", "judul", "summary"]


def muat_csv(filepath: str, source_name: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        print(f"Peringatan: {filepath} tidak ditemukan. Mengabaikan file ini.")
        return pd.DataFrame(columns=BASE_COLS + ["source"])

    df = pd.read_csv(filepath, engine="python", on_bad_lines="skip")
    for col in BASE_COLS:
        if col not in df.columns:
            df[col] = "" if col != "hoax" else np.nan

    df = df[BASE_COLS].copy()
    df["source"] = source_name
    return df


def norm_str(value: str) -> str:
    return str(value).strip().lower()


def batasi_kata(text: str, max_words: int) -> str:
    words = str(text).split()
    if len(words) <= max_words:
        return " ".join(words).strip()
    return " ".join(words[:max_words]).strip()


def ambil_fallback_text(row: pd.Series) -> str:
    for col in ["Clean Narasi", "Narasi", "isi_berita", "judul", "summary"]:
        val = row.get(col, "")
        if isinstance(val, str) and val.strip():
            return val.strip()
    return ""


def bangun_text_dan_strategy(row: pd.Series) -> Tuple[str, str]:
    judul = str(row.get("judul", "")).strip()
    summary = str(row.get("summary", "")).strip()

    if cfg.text_strategy == "judul_summary":
        if judul and summary and len(summary.split()) >= cfg.summary_min_words:
            summary_ringkas = batasi_kata(summary, cfg.summary_max_words)
            return f"{judul} [SEP] {summary_ringkas}".strip(), "judul_summary"

    sumber = ambil_fallback_text(row)
    if not sumber:
        return "", "fallback"

    kalimat = sent_tokenize(sumber)
    ringkas = " ".join(kalimat[: cfg.fallback_num_sentences]).strip()
    if not ringkas:
        ringkas = sumber
    return batasi_kata(ringkas, cfg.fallback_max_words), "fallback"


def cetak_ringkasan_dataset(df_raw: pd.DataFrame):
    if df_raw.empty:
        print("Ringkasan dataset: Data kosong.")
        return

    df_audit = df_raw.copy()
    df_audit["hoax_num"] = pd.to_numeric(df_audit["hoax"], errors="coerce")
    mask_nan = df_audit["hoax_num"].isna()
    df_audit.loc[(df_audit["source"].isin(["cnn", "detik", "kompas"])) & mask_nan, "hoax_num"] = 0
    df_audit.loc[(df_audit["source"] == "turnbackhoax") & mask_nan, "hoax_num"] = 1

    print("=== Audit Dataset Mentah ===")
    print(f"Total baris gabungan: {len(df_audit)}")

    sumber_urut = sorted(df_audit["source"].astype(str).unique())
    for sumber in sumber_urut:
        df_src = df_audit[df_audit["source"] == sumber].copy()
        print("")
        print(f"[Sumber: {sumber}] baris={len(df_src)}")
        label_dist = (
            df_src["hoax_num"].dropna().astype(int).value_counts().sort_index().to_dict()
        )
        print(f"Distribusi label hoax: {label_dist}")
        print("Kolom | dtype | missing_rate%")
        for col in BASE_COLS:
            dtype_col = str(df_src[col].dtype)
            miss_rate = float(df_src[col].isna().mean() * 100)
            print(f"- {col} | {dtype_col} | {miss_rate:.2f}")

        print("Statistik panjang teks (karakter):")
        for col in TEXT_LEN_CANDIDATES:
            seri = df_src[col].fillna("").astype(str)
            panjang = seri.str.len()
            print(
                f"- {col}: mean={panjang.mean():.1f}, "
                f"p50={panjang.quantile(0.5):.1f}, "
                f"p90={panjang.quantile(0.9):.1f}, "
                f"p95={panjang.quantile(0.95):.1f}, "
                f"max={int(panjang.max())}"
            )

    total_label = (
        df_audit["hoax_num"].dropna().astype(int).value_counts().sort_index().to_dict()
    )
    print("")
    print(f"Distribusi label total: {total_label}")


def proses_data(df_raw: pd.DataFrame):
    if df_raw.empty:
        return pd.DataFrame(columns=["text", "label"]), {}

    df = df_raw.copy()
    stats = {"baris_awal": int(len(df))}

    # Normalisasi label
    df["hoax_num"] = pd.to_numeric(df["hoax"], errors="coerce")
    mask_nan = df["hoax_num"].isna()
    df.loc[(df["source"].isin(["cnn", "detik", "kompas"])) & mask_nan, "hoax_num"] = 0
    df.loc[(df["source"] == "turnbackhoax") & mask_nan, "hoax_num"] = 1
    df = df[df["hoax_num"].isin([0, 1])].copy()
    df["label"] = df["hoax_num"].astype(int)

    # Dedup bertahap
    if cfg.dedup_strategy == "url_then_judul_tanggal_then_text_label":
        if "url" in df.columns:
            df["url_norm"] = df["url"].map(norm_str)
            baris_sebelum = len(df)
            df = df.drop_duplicates(subset=["url_norm"], keep="first")
            stats["duplikat_url"] = int(baris_sebelum - len(df))
        stats["setelah_dedup_url"] = int(len(df))

        if all(col in df.columns for col in ["judul", "tanggal"]):
            df["judul_norm"] = df["judul"].map(norm_str)
            df["tanggal_norm"] = df["tanggal"].map(norm_str)
            baris_sebelum = len(df)
            df = df.drop_duplicates(subset=["judul_norm", "tanggal_norm"], keep="first")
            stats["duplikat_judul_tanggal"] = int(baris_sebelum - len(df))
        stats["setelah_dedup_judul_tanggal"] = int(len(df))

    # Bentuk text utama
    text_strategy = df.apply(bangun_text_dan_strategy, axis=1, result_type="expand")
    df["text"] = text_strategy[0]
    df["text_strategy"] = text_strategy[1]

    df["text"] = df["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    df = df[df["text"] != ""].copy()
    stats["text_strategy_counts"] = df["text_strategy"].value_counts().to_dict()

    # Dedup akhir berdasarkan text+label
    baris_sebelum = len(df)
    df = df.drop_duplicates(subset=["text", "label"], keep="first").reset_index(drop=True)
    stats["duplikat_text_label"] = int(baris_sebelum - len(df))
    stats["setelah_dedup_text_label"] = int(len(df))
    stats["label_final"] = df["label"].value_counts().sort_index().to_dict()

    return df[["text", "label"]], stats


def chunk_list(data_list: List[str], size: int):
    for idx in range(0, len(data_list), size):
        yield data_list[idx : idx + size]


def bersihkan_token_entitas(token: str) -> str:
    token = str(token).replace("##", "").strip()
    token = re.sub(r"\s+", " ", token)
    token = re.sub(r"^[\W_]+|[\W_]+$", "", token, flags=re.UNICODE)
    return token.strip()


def ekstrak_entitas_berguna(entitas_mentah: List[Dict]) -> List[Dict]:
    hasil = []
    for ent in entitas_mentah:
        ent_type = str(ent.get("entity_group", "")).strip().upper()
        kata = bersihkan_token_entitas(ent.get("word", ""))
        if not kata:
            continue
        if ent_type in cfg.augment_entity_types:
            hasil.append({"word": kata, "type": ent_type})
    return hasil


def build_entity_pool_from_minority(texts: List[str]):
    pool = defaultdict(list)
    entities_per_text = [[] for _ in texts]

    if len(texts) == 0:
        return pool, entities_per_text

    ner_device = 0 if torch.cuda.is_available() else -1
    chunk_size = 32 if ner_device == 0 else 8

    try:
        ner_aug_pipeline = pipeline(
            "ner",
            model=cfg.ner_model_name,
            aggregation_strategy="simple",
            device=ner_device,
        )
    except Exception as e:
        print(f"Peringatan: NER augmentasi gagal dimuat ({e}). Hanya fallback EDA yang digunakan.")
        return pool, entities_per_text

    offset = 0
    for text_chunk in chunk_list(texts, chunk_size):
        try:
            hasil_chunk = ner_aug_pipeline(text_chunk, batch_size=chunk_size)
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and ner_device == 0:
                print("Peringatan: OOM saat NER augmentasi di GPU, beralih ke CPU.")
                del ner_aug_pipeline
                torch.cuda.empty_cache()
                gc.collect()
                ner_device = -1
                chunk_size = 8
                ner_aug_pipeline = pipeline(
                    "ner",
                    model=cfg.ner_model_name,
                    aggregation_strategy="simple",
                    device=ner_device,
                )
                hasil_chunk = ner_aug_pipeline(text_chunk, batch_size=chunk_size)
            else:
                raise

        for local_idx, entitas in enumerate(hasil_chunk):
            global_idx = offset + local_idx
            parsed_entities = ekstrak_entitas_berguna(entitas if isinstance(entitas, list) else [])
            entities_per_text[global_idx] = parsed_entities
            for ent in parsed_entities:
                pool[ent["type"]].append(ent["word"])

        offset += len(text_chunk)

    for ent_type in list(pool.keys()):
        pool[ent_type] = list(dict.fromkeys(pool[ent_type]))

    del ner_aug_pipeline
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return pool, entities_per_text


def replace_entities_same_type(
    text: str, entities: List[Dict], entity_pool: Dict[str, List[str]], rng: random.Random
):
    hasil = text
    replaced = 0

    kandidat = [
        ent for ent in entities if ent.get("type") in entity_pool and len(entity_pool[ent.get("type")]) > 1
    ]
    rng.shuffle(kandidat)

    for ent in kandidat:
        if replaced >= cfg.augment_max_per_sample:
            break

        kata_asli = ent.get("word", "")
        tipe = ent.get("type", "")
        opsi_pengganti = [w for w in entity_pool.get(tipe, []) if norm_str(w) != norm_str(kata_asli)]
        if not opsi_pengganti:
            continue

        kata_baru = rng.choice(opsi_pengganti)
        pattern = re.compile(rf"(?<!\w){re.escape(kata_asli)}(?!\w)", flags=re.IGNORECASE)
        hasil_baru, jumlah_replace = pattern.subn(kata_baru, hasil, count=1)
        if jumlah_replace > 0:
            hasil = hasil_baru
            replaced += 1

    return hasil, replaced > 0


def fallback_eda_ringan(text: str, rng: random.Random, prob: float) -> str:
    words = str(text).split()
    if len(words) < 8:
        return text

    hasil = words[:]

    if rng.random() < prob and len(hasil) > 8:
        idx = rng.randrange(1, len(hasil) - 1)
        del hasil[idx]

    if rng.random() < prob and len(hasil) > 8:
        idx = rng.randrange(0, len(hasil) - 1)
        hasil[idx], hasil[idx + 1] = hasil[idx + 1], hasil[idx]

    return " ".join(hasil).strip()


def augment_minority_train_only(df_train: pd.DataFrame):
    if not cfg.augment_enable:
        return df_train.copy(), {"augmented_total": 0, "augmented_ner": 0, "augmented_eda": 0}

    counts = df_train["label"].value_counts()
    if len(counts) < 2:
        return df_train.copy(), {"augmented_total": 0, "augmented_ner": 0, "augmented_eda": 0}

    label_mayor = int(counts.idxmax())
    label_minor = int(counts.idxmin())

    target_minor = int(counts[label_mayor] * cfg.augment_target_ratio)
    gap = max(0, target_minor - int(counts[label_minor]))
    if gap == 0:
        return df_train.copy(), {"augmented_total": 0, "augmented_ner": 0, "augmented_eda": 0}

    df_minor = df_train[df_train["label"] == label_minor].reset_index(drop=True)
    texts_minor = df_minor["text"].tolist()

    entity_pool, entities_per_text = build_entity_pool_from_minority(texts_minor)

    rng = random.Random(cfg.augment_seed)
    indeks = list(range(len(texts_minor)))
    augmented_records = []
    augmented_pairs = set()
    existing_pairs = set((str(t), int(l)) for t, l in zip(df_train["text"], df_train["label"]))
    augmented_ner = 0
    augmented_eda = 0
    max_rounds = max(1, int(cfg.augment_max_rounds))

    for _ in range(max_rounds):
        if len(augmented_records) >= gap:
            break

        rng.shuffle(indeks)
        for idx in indeks:
            if len(augmented_records) >= gap:
                break

            text_src = texts_minor[idx]
            entitas = entities_per_text[idx] if idx < len(entities_per_text) else []

            text_aug, success_ner = replace_entities_same_type(text_src, entitas, entity_pool, rng)
            if success_ner and text_aug and text_aug != text_src:
                pair = (text_aug, label_minor)
                if pair not in existing_pairs and pair not in augmented_pairs:
                    augmented_records.append({"text": text_aug, "label": label_minor})
                    augmented_pairs.add(pair)
                    augmented_ner += 1
                    continue

            text_aug_eda = fallback_eda_ringan(text_src, rng, cfg.augment_fallback_eda_prob)
            if text_aug_eda and text_aug_eda != text_src:
                pair = (text_aug_eda, label_minor)
                if pair not in existing_pairs and pair not in augmented_pairs:
                    augmented_records.append({"text": text_aug_eda, "label": label_minor})
                    augmented_pairs.add(pair)
                    augmented_eda += 1

    if len(augmented_records) == 0:
        return df_train.copy(), {"augmented_total": 0, "augmented_ner": 0, "augmented_eda": 0}

    df_aug = pd.DataFrame(augmented_records)
    df_train_aug = pd.concat([df_train, df_aug], ignore_index=True)
    df_train_aug = df_train_aug.drop_duplicates(subset=["text", "label"], keep="first").reset_index(drop=True)

    stats_aug = {
        "augmented_total": int(len(df_train_aug) - len(df_train)),
        "augmented_ner": int(augmented_ner),
        "augmented_eda": int(augmented_eda),
        "label_minor": label_minor,
        "label_mayor": label_mayor,
        "target_minor": target_minor,
    }
    return df_train_aug, stats_aug


# Eksekusi pembacaan file dari Google Drive
frames = [
    muat_csv(os.path.join(cfg.data_dir, cfg.file_hoaks), "turnbackhoax"),
    muat_csv(os.path.join(cfg.data_dir, cfg.file_nonhoaks_cnn), "cnn"),
    muat_csv(os.path.join(cfg.data_dir, cfg.file_nonhoaks_detik), "detik"),
    muat_csv(os.path.join(cfg.data_dir, cfg.file_nonhoaks_kompas), "kompas"),
]
df_mentah = pd.concat(frames, ignore_index=True)
cetak_ringkasan_dataset(df_mentah)

df_final, stats_prep = proses_data(df_mentah)

if df_final.empty:
    print("DATASET KOSONG: Pastikan path Google Drive Anda benar. Menggunakan data dummy...")
    df_final = pd.DataFrame({
        "text": ["Vaksin mengandung mikrocip", "Bantuan pemerintah cair hari ini", "Berita ini palsu", "Fakta data BPS"],
        "label": [1, 0, 1, 0],
    })

# Mempartisi data asli murni (augmentasi hanya dilakukan setelah split, pada set latih)
df_latih, df_sementara = train_test_split(
    df_final,
    test_size=0.30,
    stratify=df_final["label"],
    random_state=cfg.seed,
)
df_validasi, df_uji = train_test_split(
    df_sementara,
    test_size=0.50,
    stratify=df_sementara["label"],
    random_state=cfg.seed,
)

label_validasi_awal = df_validasi["label"].copy(deep=True)
label_uji_awal = df_uji["label"].copy(deep=True)

ukuran_validasi_awal = len(df_validasi)
ukuran_uji_awal = len(df_uji)

distribusi_train_awal = df_latih["label"].value_counts().to_dict()
df_latih, stats_aug = augment_minority_train_only(df_latih)
distribusi_train_akhir = df_latih["label"].value_counts().to_dict()

assert len(df_validasi) == ukuran_validasi_awal, "Validasi berubah ukuran; augmentasi harus train-only."
assert len(df_uji) == ukuran_uji_awal, "Uji berubah ukuran; augmentasi harus train-only."
assert df_validasi["label"].equals(label_validasi_awal), "Label validasi berubah; augmentasi harus train-only."
assert df_uji["label"].equals(label_uji_awal), "Label uji berubah; augmentasi harus train-only."

print("=== Ringkasan Prapemrosesan ===")
print(f"Jumlah baris awal gabungan: {len(df_mentah)}")
print(f"Duplikat URL dihapus: {stats_prep.get('duplikat_url', 0)}")
print(f"Setelah dedup URL: {stats_prep.get('setelah_dedup_url', 'N/A')}")
print(f"Duplikat judul+tanggal dihapus: {stats_prep.get('duplikat_judul_tanggal', 0)}")
print(f"Setelah dedup judul+tanggal: {stats_prep.get('setelah_dedup_judul_tanggal', 'N/A')}")
print(f"Duplikat text+label dihapus: {stats_prep.get('duplikat_text_label', 0)}")
print(f"Setelah dedup text+label: {stats_prep.get('setelah_dedup_text_label', 'N/A')}")
print(f"Distribusi label final: {stats_prep.get('label_final', {})}")
print(f"Strategi representasi text: {stats_prep.get('text_strategy_counts', {})}")
print(f"Total data final: {len(df_final)}")

print("")
print("=== Split Dataset ===")
print(f"Latih: {len(df_latih)} | Validasi: {len(df_validasi)} | Uji: {len(df_uji)}")

print("")
print("=== Distribusi Label Train ===")
print(f"Sebelum augmentasi: {distribusi_train_awal}")
print(f"Sesudah augmentasi: {distribusi_train_akhir}")

print("")
print("=== Ringkasan Augmentasi Minoritas (Train Only) ===")
print(f"Augmented total: {stats_aug.get('augmented_total', 0)}")
print(f"Augmented via NER replacement: {stats_aug.get('augmented_ner', 0)}")
print(f"Augmented via fallback EDA: {stats_aug.get('augmented_eda', 0)}")


=== Audit Dataset Mentah ===
Total baris gabungan: 37007

[Sumber: cnn] baris=807
Distribusi label hoax: {0: 807}
Kolom | dtype | missing_rate%
- url | object | 0.00
- judul | object | 0.00
- tanggal | object | 0.00
- isi_berita | object | 0.00
- Narasi | object | 0.00
- Clean Narasi | object | 0.00
- hoax | int64 | 0.00
- summary | object | 0.00
Statistik panjang teks (karakter):
- Clean Narasi: mean=5814.2, p50=5448.0, p90=8542.4, p95=10449.1, max=67069
- Narasi: mean=8146.5, p50=7407.0, p90=12365.6, p95=15095.6, max=111187
- isi_berita: mean=8146.5, p50=7407.0, p90=12365.6, p95=15095.6, max=111187
- judul: mean=59.8, p50=62.0, p90=69.0, p95=70.0, max=70
- summary: mean=388.2, p50=377.0, p90=546.0, p95=576.7, max=2757

[Sumber: detik] baris=18711
Distribusi label hoax: {0: 18711}
Kolom | dtype | missing_rate%
- url | object | 0.00
- judul | object | 0.00
- tanggal | object | 0.00
- isi_berita | object | 0.00
- Narasi | object | 0.00
- Clean Narasi | object | 0.00
- hoax | int64 | 0.0

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== Ringkasan Prapemrosesan ===
Jumlah baris awal gabungan: 37007
Duplikat URL dihapus: 0
Setelah dedup URL: 37007
Duplikat judul+tanggal dihapus: 42
Setelah dedup judul+tanggal: 36965
Duplikat text+label dihapus: 45
Setelah dedup text+label: 36920
Distribusi label final: {0: 24599, 1: 12321}
Strategi representasi text: {'judul_summary': 36764, 'fallback': 201}
Total data final: 36920

=== Split Dataset ===
Latih: 34438 | Validasi: 5538 | Uji: 5538

=== Distribusi Label Train ===
Sebelum augmentasi: {0: 17219, 1: 8625}
Sesudah augmentasi: {1: 17219, 0: 17219}

=== Ringkasan Augmentasi Minoritas (Train Only) ===
Augmented total: 8594
Augmented via NER replacement: 8479
Augmented via fallback EDA: 115


In [6]:
# 6. Tokenisasi Dataset Hugging Face
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)


def tokenisasi_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=cfg.max_length)


ds_latih = Dataset.from_pandas(df_latih, preserve_index=False)
ds_validasi = Dataset.from_pandas(df_validasi, preserve_index=False)
ds_uji = Dataset.from_pandas(df_uji, preserve_index=False)

ds_latih = ds_latih.map(tokenisasi_batch, batched=True)
ds_validasi = ds_validasi.map(tokenisasi_batch, batched=True)
ds_uji = ds_uji.map(tokenisasi_batch, batched=True)

kolom_format = ["input_ids", "attention_mask", "label"]
ds_latih.set_format(type="torch", columns=kolom_format)
ds_validasi.set_format(type="torch", columns=kolom_format)
ds_uji.set_format(type="torch", columns=kolom_format)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if device == "cuda" else None,
)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/34438 [00:00<?, ? examples/s]

Map:   0%|          | 0/5538 [00:00<?, ? examples/s]

Map:   0%|          | 0/5538 [00:00<?, ? examples/s]

In [7]:
# 7. Inisialisasi Model IndoBERT dan Pelatihan (VRAM Teroptimasi)
model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=2,
    use_safetensors=False,  # Menghindari error latar belakang
)
model.to(device)
model.config.use_cache = False  # direkomendasikan saat gradient checkpointing aktif


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(p_weighted),
        "recall_weighted": float(r_weighted),
        "f1_weighted": float(f1_weighted),
        "cm_tn": int(tn),
        "cm_fp": int(fp),
        "cm_fn": int(fn),
        "cm_tp": int(tp),
    }


train_counts = df_latih["label"].value_counts().sort_index()
n_train = float(len(df_latih))
num_classes = 2
class_weights = []
for class_id in range(num_classes):
    count_class = float(train_counts.get(class_id, 1.0))
    class_weights.append(n_train / (num_classes * count_class))

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
if device == "cuda":
    class_weights_tensor = class_weights_tensor.to(device)

print(f"Class weights (0,1): {class_weights}")
print(f"Imbalance method: {cfg.imbalance_method}")


class ImbalanceTrainer(Trainer):
    def __init__(self, *args, class_weights=None, cfg_obj=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.cfg_obj = cfg_obj

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.cfg_obj.imbalance_method == "focal":
            ce_loss = F.cross_entropy(logits, labels, weight=self.class_weights, reduction="none")
            pt = torch.exp(-ce_loss)
            loss = ((1 - pt) ** self.cfg_obj.focal_gamma * ce_loss).mean()
        else:
            loss = F.cross_entropy(logits, labels, weight=self.class_weights)

        return (loss, outputs) if return_outputs else loss


argumen_kwargs = dict(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.train_batch_size,
    per_device_eval_batch_size=cfg.eval_batch_size,
    gradient_accumulation_steps=cfg.grad_accumulation,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    num_train_epochs=cfg.num_epochs,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    fp16=True if device == "cuda" else False,
    gradient_checkpointing=True,
    auto_find_batch_size=cfg.auto_find_batch_size,
    dataloader_num_workers=2,
    report_to="none",
)

training_arg_names = TrainingArguments.__init__.__code__.co_varnames
if "eval_strategy" in training_arg_names:
    argumen_kwargs["eval_strategy"] = "epoch"
else:
    argumen_kwargs["evaluation_strategy"] = "epoch"

argumen_latih = TrainingArguments(**argumen_kwargs)

trainer_kwargs = dict(
    model=model,
    args=argumen_latih,
    train_dataset=ds_latih,
    eval_dataset=ds_validasi,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor,
    cfg_obj=cfg,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=cfg.early_stopping_patience,
            early_stopping_threshold=cfg.early_stopping_threshold,
        )
    ],
)

trainer_arg_names = Trainer.__init__.__code__.co_varnames
if "processing_class" in trainer_arg_names:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = ImbalanceTrainer(**trainer_kwargs)

print("Memulai pelatihan (dengan augmentasi train-only + metrik lengkap)...")
trainer.train()

metric_validasi = trainer.evaluate(eval_dataset=ds_validasi)
metric_uji = trainer.evaluate(eval_dataset=ds_uji, metric_key_prefix="test")

print("")
print("=== Metrik Validasi ===")
for k, v in metric_validasi.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

print("")
print("=== Metrik Uji ===")
for k, v in metric_uji.items():
    if isinstance(v, (int, float)):
        print(f"{k}: {v}")

trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)


pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 47, in spawn_con

Class weights (0,1): [1.0, 1.0]
Imbalance method: class_weight
Memulai pelatihan (dengan augmentasi train-only + metrik lengkap)...


Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted,Cm Tn,Cm Fp,Cm Fn,Cm Tp
1,0.003352,0.000008,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3690,0,0,1848
2,0.001730,0.000005,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3690,0,0,1848
3,0.000010,0.000003,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3690,0,0,1848


...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled



=== Metrik Validasi ===
eval_loss: 7.998592082003597e-06
eval_accuracy: 1.0
eval_precision_macro: 1.0
eval_recall_macro: 1.0
eval_f1_macro: 1.0
eval_precision_weighted: 1.0
eval_recall_weighted: 1.0
eval_f1_weighted: 1.0
eval_cm_tn: 3690
eval_cm_fp: 0
eval_cm_fn: 0
eval_cm_tp: 1848
eval_runtime: 14.1752
eval_samples_per_second: 390.682
eval_steps_per_second: 12.275
epoch: 3.0

=== Metrik Uji ===
test_loss: 7.969898433657363e-06
test_accuracy: 1.0
test_precision_macro: 1.0
test_recall_macro: 1.0
test_f1_macro: 1.0
test_precision_weighted: 1.0
test_recall_weighted: 1.0
test_f1_weighted: 1.0
test_cm_tn: 3690
test_cm_fp: 0
test_cm_fn: 0
test_cm_tp: 1848
test_runtime: 14.9459
test_samples_per_second: 370.535
test_steps_per_second: 11.642
epoch: 3.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('indobert_hoax_ner_model/tokenizer_config.json',
 'indobert_hoax_ner_model/tokenizer.json')

In [8]:
# 8. Fitur Final: Analisis Simultan - Klasifikasi Kalimat & Ekstraksi Entitas (NER)
print("Mempersiapkan modul Named Entity Recognition (NER) sekunder...")

ner_device = cfg.ner_infer_device_default
device_klasifikasi = device

if cfg.ner_infer_use_gpu and torch.cuda.is_available():
    ner_device = 0
    if cfg.ner_safe_offload_classifier:
        model.to("cpu")
        device_klasifikasi = "cpu"
        torch.cuda.empty_cache()
        gc.collect()
        print("Model klasifikasi dipindah ke CPU sebelum memuat NER di GPU (mode aman).")
else:
    # Default tetap CPU agar menghindari OOM saat inferensi NER
    ner_device = -1

if ner_device >= 0 and not torch.cuda.is_available():
    ner_device = -1

ner_pipeline = pipeline(
    "ner",
    model=cfg.ner_model_name,
    aggregation_strategy="simple",
    device=ner_device,
)

print(f"Device klasifikasi: {device_klasifikasi} | Device NER pipeline: {ner_device}")


def prediksi_kalimat_dan_entitas(teks_artikel: str, model_klasifikasi, tokenizer_model, target_device: str) -> List[Dict]:
    """
    Membelah paragraf menjadi kalimat. Mengevaluasi status hoaks tiap kalimat,
    lalu mengekstrak entitas penting (Orang, Lokasi, Organisasi) dari dalamnya.
    """
    daftar_kalimat = sent_tokenize(teks_artikel)
    hasil_analisis = []

    model_klasifikasi.eval()
    for kalimat in daftar_kalimat:
        kalimat = kalimat.strip()
        if not kalimat:
            continue

        # Tahap 1: Evaluasi Polaritas Fakta/Hoaks via IndoBERT Custom
        inputs = tokenizer_model(
            kalimat,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
        ).to(target_device)

        with torch.no_grad():
            outputs = model_klasifikasi(**inputs)
            probabilitas = torch.softmax(outputs.logits, dim=1).squeeze()

            prediksi_kelas = torch.argmax(probabilitas).item()
            skor_keyakinan = probabilitas[prediksi_kelas].item()
            label_final = "Hoaks" if prediksi_kelas == 1 else "Fakta"

        # Tahap 2: Ekstraksi Entitas Khusus via Pipeline NER
        entitas_mentah = ner_pipeline(kalimat)
        daftar_entitas = []
        for ent in entitas_mentah:
            # Hanya ambil entitas dengan confidence > 60% untuk mengurangi noise
            if ent["score"] > 0.60:
                daftar_entitas.append({
                    "kata": ent["word"],
                    "jenis": ent["entity_group"],
                    "akurasi_ner": round(ent["score"] * 100, 2),
                })

        hasil_analisis.append({
            "kalimat": kalimat,
            "label_hoaks": label_final,
            "probabilitas_hoaks": round(skor_keyakinan * 100, 2),
            "entitas_ditemukan": daftar_entitas,
        })

    return hasil_analisis


# ----------------- BLOK PENGUJIAN -----------------
artikel_uji = (
    "Gubernur DKI Jakarta Anies Baswedan dikabarkan membagikan uang tunai sebesar 5 juta rupiah "
    "melalui tautan WhatsApp ini. Namun faktanya, Kementerian Komunikasi dan Informatika (Kominfo) "
    "telah mengklarifikasi bahwa pesan tersebut adalah penipuan. Jangan pernah memberikan data "
    "pribadi Anda kepada pihak yang mengatasnamakan Pemprov DKI Jakarta."
)

print("")
print("=== HASIL ANALISIS INTEGRASI (KLASIFIKASI HOAKS + NER) ===")
hasil_gabungan = prediksi_kalimat_dan_entitas(artikel_uji, model, tokenizer, device_klasifikasi)

for idx, item in enumerate(hasil_gabungan):
    print("")
    print(f"[Kalimat {idx + 1}] | Status: {item['label_hoaks']} ({item['probabilitas_hoaks']}%) ")
    print(f"Teks    : {item['kalimat']}")
    if item["entitas_ditemukan"]:
        print("Entitas :")
        for ent in item["entitas_ditemukan"]:
            print(f"  - {ent['kata']} ({ent['jenis']})")
    else:
        print("Entitas : Tidak mendeteksi nama/lokasi penting.")


Mempersiapkan modul Named Entity Recognition (NER) sekunder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: cahya/bert-base-indonesian-NER
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.pooler.dense.bias       | UNEXPECTED |  | 
bert.embeddings.position_ids | UNEXPECTED |  | 
bert.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device klasifikasi: cuda | Device NER pipeline: -1

=== HASIL ANALISIS INTEGRASI (KLASIFIKASI HOAKS + NER) ===

[Kalimat 1] | Status: Fakta (100.0%) 
Teks    : Gubernur DKI Jakarta Anies Baswedan dikabarkan membagikan uang tunai sebesar 5 juta rupiah melalui tautan WhatsApp ini.
Entitas :
  - gubernur dki jakarta (NOR)
  - anies baswedan (PER)
  - 5 juta rupiah (MON)
  - whatsapp (PRD)

[Kalimat 2] | Status: Fakta (100.0%) 
Teks    : Namun faktanya, Kementerian Komunikasi dan Informatika (Kominfo) telah mengklarifikasi bahwa pesan tersebut adalah penipuan.
Entitas :
  - kementerian komunikasi dan informatika (NOR)
  - kominfo (NOR)

[Kalimat 3] | Status: Fakta (100.0%) 
Teks    : Jangan pernah memberikan data pribadi Anda kepada pihak yang mengatasnamakan Pemprov DKI Jakarta.
Entitas :
  - pemprov dki jakarta (NOR)


In [11]:
drive.mount('/content/drive')
folder_tujuan = '/content/drive/MyDrive/berita_hoax_output'
os.makedirs(folder_tujuan, exist_ok=True)

nama_file_zip = "indobert_hoax_ner_model_final"
nama_file_zip_lengkap = f"{nama_file_zip}.zip"

print(f"Sedang mengompresi direktori {cfg.output_dir}...")
shutil.make_archive(nama_file_zip, 'zip', cfg.output_dir)

jalur_sumber = nama_file_zip_lengkap
jalur_target = os.path.join(folder_tujuan, nama_file_zip_lengkap)

print(f"Mengunggah file {nama_file_zip_lengkap} ke Google Drive di folder 'berita_hoax_output'...")
try:
    shutil.copy(jalur_sumber, jalur_target)
    print(f"Proses unggah berhasil. File tersimpan di: {jalur_target}")
except Exception as e:
    print(f"Terjadi kesalahan saat mengunggah ke Google Drive: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sedang mengompresi direktori indobert_hoax_ner_model...
Mengunggah file indobert_hoax_ner_model_final.zip ke Google Drive di folder 'berita_hoax_output'...
Proses unggah berhasil. File tersimpan di: /content/drive/MyDrive/berita_hoax_output/indobert_hoax_ner_model_final.zip


In [12]:
!pip install -q huggingface_hub

from huggingface_hub import login, HfApi
from google.colab import userdata

try:
    token_kredensial = userdata.get('HF_TOKEN')
    login(token=token_kredensial)
except Exception:
    print("Token HF_TOKEN tidak ditemukan pada Colab Secrets.")

api_huggingface = HfApi()
id_repositori_target = "fjrmhri/Deteksi_Hoax_TA"
direktori_sumber_artefak = cfg.output_dir

api_huggingface.create_repo(repo_id=id_repositori_target, private=False, exist_ok=True)

print(f"Memulai unggahan dari {direktori_sumber_artefak} menuju {id_repositori_target}...")
api_huggingface.upload_folder(
    folder_path=direktori_sumber_artefak,
    repo_id=id_repositori_target,
    repo_type="model",
    commit_message="Pembaruan model IndoBERT deteksi hoaks"
)
print(f"Transmisi selesai. URL Repositori: https://huggingface.co/{id_repositori_target}")

Memulai unggahan dari indobert_hoax_ner_model menuju fjrmhri/Deteksi_Hoax_TA...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-3231/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...kpoint-1077/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  ...r_model/model.safetensors:   0%|          | 92.7kB /  442MB            

  ...nt-3231/model.safetensors:   0%|          |  560kB /  442MB            

  ...ckpoint-1077/scheduler.pt:   6%|5         |  86.0B / 1.47kB            

  ...nt-1077/training_args.bin:   6%|5         |   306B / 5.20kB            

Transmisi selesai. URL Repositori: https://huggingface.co/fjrmhri/Deteksi_Hoax_TA
